In [3]:
import sys
import os
sys.path.append(os.path.abspath(".."))
os.chdir("..") 
print("CWD:", os.getcwd()) 

CWD: /Users/oliver/Desktop/BitcoinGraphClassification


In [ ]:
import os
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from data_loader import load_wallet_graph_combined  


data_dir = os.path.abspath(os.path.join(".", "data", "raw"))



data = load_wallet_graph_combined(data_dir)

x = data.x.cpu().numpy()
y = data.y.cpu().numpy()

mask = y >= 0
x = x[mask]
y = y[mask]

feature_file = os.path.join(data_dir, "wallets_features_classes_combined.csv")
df_raw = pd.read_csv(feature_file)
cols_to_drop = [col for col in ["address", "class", "node_id"] if col in df_raw.columns]
feature_names = df_raw.drop(columns=cols_to_drop).columns.tolist()

df_features = pd.DataFrame(x, columns=feature_names)
df_features["class"] = ["licit" if label == 0 else "illicit" for label in y]

transactional = [col for col in feature_names if "btc_" in col or "fees" in col]
temporal = [col for col in feature_names if "block" in col or "timesteps" in col]
structural = [
    "num_txs_as_sender", "num_txs_as receiver",
    "num_addr_transacted_multiple", "transacted_w_address_total",
    "transacted_w_address_mean", "transacted_w_address_median"
]

feature_sets = {
    "Transactional": transactional,
    "Temporal": temporal,
    "Structural": structural
}

scaler = MinMaxScaler()

for category, features in feature_sets.items():
    subset = df_features[features + ["class"]].dropna()
    subset_scaled = pd.DataFrame(
        scaler.fit_transform(subset[features]),
        columns=features
    )
    subset_scaled["class"] = subset["class"].values

    melted = subset_scaled.melt(id_vars="class", var_name="feature", value_name="value")

    plt.figure(figsize=(18, 6))
    sns.boxplot(data=melted, x="feature", y="value", hue="class")
    plt.xticks(rotation=90)
    plt.title(f"{category} Features – Normalized Distributions by Class")
    plt.tight_layout()
    plt.grid(True)
    plt.savefig(f"{category.lower()}_features_boxplot.png")
    plt.close()
